# Ames Housing price prediction

In this notebook, we work on the Ames Housing regression challenge. Our goal is to predict `SalePrice` for houses in the test set.

We use a complete machine learning workflow: we load the data, do some exploratory analysis, create additional features, handle missing values, remove one clear extreme outlier from the training data, tune models with cross-validation, and finally generate a CSV file with predictions.

The main model we focus on is XGBoost, because gradient boosting models often perform very well on tabular regression problems like housing-price prediction.


## Importing libraries

In this cell, we import the libraries we need for the notebook.

We use `pandas` and `numpy` for data manipulation, `matplotlib` for plots, and `sklearn` for preprocessing, pipelines, validation, and baseline models. We also import `XGBRegressor` from XGBoost, which is the main model we test after the baseline models.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.ticker import FuncFormatter

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from xgboost import XGBRegressor

from IPython.display import display


## Loading the data

In this cell, we load the training features, the training target, and the test features.

We keep the `Id` column from the test set in a separate variable because we need it later for the submission file. For modelling, we remove `Id` from both training and test features because it is only an identifier, not a real explanatory variable.

We also convert `y_train` from a one-column dataframe to a pandas Series containing only `SalePrice`.


In [ ]:
X_train = pd.read_csv("ames_X_train.csv")
y_train = pd.read_csv("ames_y_train.csv")["SalePrice"]
X_test = pd.read_csv("ames_X_test.csv")

test_ids = X_test["Id"].copy()

X_train = X_train.drop(columns=["Id"])
X_test = X_test.drop(columns=["Id"])

print("Training features shape:", X_train.shape)
print("Training target shape:", y_train.shape)
print("Test features shape:", X_test.shape)

display(X_train.head())


## Creating new Ames-specific features

In this cell, we define a function that creates additional variables from the original Ames columns.

We create `TotalSF`, `TotalBath`, `TotalPorchSF`, `HouseAge`, and `RemodAge`. These features summarize information that is already present in the dataset. For example, total square footage is usually more directly related to house price than looking at basement, first floor, and second floor area separately.

This does not create data leakage because we only use feature values from each row. We do not use `SalePrice`, and we do not calculate statistics from the test set.

For area and bathroom totals, we use `fillna(0)` because missing values in these columns often mean that the house does not have that specific component.


In [ ]:
def add_ames_features_manual(X):
    X = X.copy()

    X["TotalSF"] = (
        X["TotalBsmtSF"].fillna(0) +
        X["1stFlrSF"].fillna(0) +
        X["2ndFlrSF"].fillna(0)
    )

    X["TotalBath"] = (
        X["FullBath"].fillna(0) +
        0.5 * X["HalfBath"].fillna(0) +
        X["BsmtFullBath"].fillna(0) +
        0.5 * X["BsmtHalfBath"].fillna(0)
    )

    X["TotalPorchSF"] = (
        X["OpenPorchSF"].fillna(0) +
        X["EnclosedPorch"].fillna(0) +
        X["3SsnPorch"].fillna(0) +
        X["ScreenPorch"].fillna(0) +
        X["WoodDeckSF"].fillna(0)
    )

    X["HouseAge"] = X["YrSold"] - X["YearBuilt"]
    X["RemodAge"] = X["YrSold"] - X["YearRemodAdd"]

    return X

X_train = add_ames_features_manual(X_train)
X_test = add_ames_features_manual(X_test)

print("Training shape after feature engineering:", X_train.shape)
print("Test shape after feature engineering:", X_test.shape)


## Removing one extreme training outlier

In this cell, we remove one extreme observation from the training data.

The condition is `GrLivArea > 4000` and `SalePrice < 300000`. This corresponds to a house with a very large living area but a surprisingly low sale price. This kind of point can heavily affect RMSE because RMSE penalizes large errors strongly.

We remove this observation only from the training data. We never remove rows from the test set because the test set is exactly what we need to predict.


In [ ]:
outlier_mask = (X_train["GrLivArea"] > 4000) & (y_train < 300000)

print("Number of removed training outliers:", outlier_mask.sum())

X_train = X_train.loc[~outlier_mask].reset_index(drop=True)
y_train = y_train.loc[~outlier_mask].reset_index(drop=True)

print("Training shape after outlier removal:", X_train.shape)
print("Target shape after outlier removal:", y_train.shape)


## Exploratory data analysis

In this cell, we inspect the dataset after feature engineering and outlier removal.

We print all feature names, then we plot some relationships with `SalePrice`. The goal is not to build the model yet, but to understand which variables seem strongly connected with house prices.


In [ ]:
for i, col in enumerate(X_train.columns, 1):
    print(f"{i}. {col}")


def euro_formatter(x, pos):
    return f"€{int(x):,}"

eda_train = X_train.copy()
eda_train["SalePrice"] = y_train

numeric_cols = X_train.select_dtypes(include=np.number).columns
corr = X_train[numeric_cols].corrwith(y_train).dropna()
top_corr = corr.abs().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top_corr.index, top_corr.values, color="tab:red", edgecolor="black")
ax.set_title("Top numeric correlations with SalePrice")
ax.set_xlabel("Absolute correlation")
ax.set_ylabel("Feature")
plt.show()

fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(eda_train["GrLivArea"], eda_train["SalePrice"], alpha=0.55, s=25, color="blue")
ax.set_title("GrLivArea vs SalePrice")
ax.set_xlabel("Living area")
ax.set_ylabel("SalePrice")
ax.yaxis.set_major_formatter(FuncFormatter(euro_formatter))
plt.show()

fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(eda_train["OverallQual"], eda_train["SalePrice"], alpha=0.45, s=25, color="red")
ax.set_title("OverallQual vs SalePrice")
ax.set_xlabel("Overall quality rating")
ax.set_ylabel("SalePrice")
ax.yaxis.set_major_formatter(FuncFormatter(euro_formatter))
plt.show()

neighborhood_order = eda_train.groupby("Neighborhood")["SalePrice"].median().sort_values().index
neighborhood_data = [
    eda_train.loc[eda_train["Neighborhood"] == neighborhood, "SalePrice"]
    for neighborhood in neighborhood_order
]

fig, ax = plt.subplots(figsize=(18, 8))
ax.boxplot(neighborhood_data, tick_labels=neighborhood_order, showfliers=False)
ax.set_title("Neighborhood vs SalePrice")
ax.set_xlabel("Neighborhood")
ax.set_ylabel("SalePrice")
ax.tick_params(axis="x", rotation=75)
ax.yaxis.set_major_formatter(FuncFormatter(euro_formatter))
plt.show()


## Separating normal and special missing-value columns

In this cell, we divide columns into numerical and categorical features.

Some Ames variables are special because a missing value often means absence. For example, if `GarageType` is missing, the house probably has no garage. For these variables, we do not want to fill missing values with the mode or the median. Instead, we fill numerical absence variables with `0`, and categorical absence variables with the string `"None"`.

The loop is written safely with `elif`, so if a column from the special list is not present, the notebook does not crash.


In [ ]:
cat_c = []
num_c = []

for col in X_train.columns:
    if X_train[col].dtype == "object":
        cat_c.append(col)
    else:
        num_c.append(col)

Particular_features = [
    "Alley",
    "FireplaceQu",
    "Fireplaces",
    "PoolArea",
    "PoolQC",
    "Fence",
    "MasVnrType",
    "MasVnrArea",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "GarageArea",
    "GarageCars",
    "GarageYrBlt",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "BsmtFullBath",
    "BsmtHalfBath",
    "TotalBsmtSF",
]

part_cat_c = []
part_num_c = []

for col in Particular_features:
    if col in num_c:
        num_c.remove(col)
        part_num_c.append(col)
    elif col in cat_c:
        cat_c.remove(col)
        part_cat_c.append(col)

print("Normal numerical columns:", len(num_c))
print("Special numerical columns filled with 0:", len(part_num_c))
print("Normal categorical columns:", len(cat_c))
print("Special categorical columns filled with 'None':", len(part_cat_c))


## Building the preprocessing transformer

In this cell, we build the preprocessing object used by all models.

Normal numerical columns are imputed with the median and scaled. Special numerical columns are imputed with `0` and scaled. Special categorical columns are imputed with `"None"`, while normal categorical columns are imputed with the most frequent category. All categorical columns are then one-hot encoded.

This preprocessing is inside a sklearn `ColumnTransformer`, so during cross-validation it is fitted only on the training part of each fold. This avoids preprocessing leakage.


In [ ]:
numerical_pipeline_median = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numerical_pipeline_null = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
    ("scaler", StandardScaler())
])

categorical_pipeline_none = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

categorical_pipeline_mode = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

MyTransformer = ColumnTransformer([
    ("numerical_median", numerical_pipeline_median, num_c),
    ("numerical_zero", numerical_pipeline_null, part_num_c),
    ("categorical_none", categorical_pipeline_none, part_cat_c),
    ("categorical_mode", categorical_pipeline_mode, cat_c),
])


## Cross-validation setup

In this cell, we define the cross-validation strategy.

We use 5-fold cross-validation with shuffling. This means the training data is split into five parts. Each model trains on four parts and validates on the remaining part, and this process repeats until every part has been used as validation once.

We use the same folds for all models, so the model comparison is fair.


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)


## Random Forest baseline with log-transformed target

In this cell, we train a Random Forest model as a baseline.

We wrap the model in `TransformedTargetRegressor`, which trains the model on `log1p(SalePrice)` and then automatically converts predictions back to the original `SalePrice` scale with `expm1`. This is useful because house prices are right-skewed.

The RMSE printed by `GridSearchCV` is still on the original SalePrice scale, because predictions are inverse-transformed before scoring.


In [ ]:
model_RFR = TransformedTargetRegressor(
    regressor=RandomForestRegressor(n_jobs=-1, random_state=42),
    func=np.log1p,
    inverse_func=np.expm1,
    check_inverse=False
)

MyPipeline_RFR = Pipeline([
    ("preprocess", MyTransformer),
    ("model", model_RFR),
])

parameters_grid_rfr = {
    "model__regressor__n_estimators": [400, 700],
    "model__regressor__max_depth": [None, 20],
    "model__regressor__min_samples_leaf": [1, 2],
    "model__regressor__min_samples_split": [2, 5],
}

Hyperparameter_tuning_RFR = GridSearchCV(
    estimator=MyPipeline_RFR,
    param_grid=parameters_grid_rfr,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    refit=True,
    n_jobs=-1,
    verbose=1,
)

Hyperparameter_tuning_RFR.fit(X_train, y_train)

print("Best Random Forest parameters:")
print(Hyperparameter_tuning_RFR.best_params_)
print("Best Random Forest CV RMSE:", -Hyperparameter_tuning_RFR.best_score_)

best_rfr = Hyperparameter_tuning_RFR.best_estimator_


## Linear Regression baseline with log-transformed target

In this cell, we train a simple linear regression baseline.

This model is less flexible than tree-based models, but it is useful as a reference point. If a complex model does not clearly beat this baseline, then something may be wrong with the modelling pipeline.


In [ ]:
model_LR = TransformedTargetRegressor(
    regressor=LinearRegression(),
    func=np.log1p,
    inverse_func=np.expm1,
    check_inverse=False
)

MyPipeline_LR = Pipeline([
    ("preprocess", MyTransformer),
    ("model", model_LR),
])

Hyperparameter_tuning_LR = GridSearchCV(
    estimator=MyPipeline_LR,
    param_grid={},
    cv=kf,
    scoring="neg_root_mean_squared_error",
    refit=True,
    n_jobs=-1,
)

Hyperparameter_tuning_LR.fit(X_train, y_train)

print("Best Linear Regression CV RMSE:", -Hyperparameter_tuning_LR.best_score_)

best_lr = Hyperparameter_tuning_LR.best_estimator_


## XGBoost model with log-transformed target

In this cell, we train the main model: XGBoost.

XGBoost builds trees sequentially, where each new tree tries to correct the errors of the previous trees. This often works very well on structured tabular data.

We again use `TransformedTargetRegressor` so the model learns on the log-transformed target but returns predictions in the original SalePrice scale. The parameter names in the grid start with `model__regressor__` because the XGBoost regressor is inside `TransformedTargetRegressor`, which is inside the pipeline.

The grid is compact enough to run in a reasonable time, but it still tests the most important XGBoost controls: number of trees, learning rate, tree depth, row sampling, column sampling, and regularization.


In [ ]:
model_XGB = TransformedTargetRegressor(
    regressor=XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
        verbosity=0,
    ),
    func=np.log1p,
    inverse_func=np.expm1,
    check_inverse=False,
)

Pipeline_XGB = Pipeline([
    ("preprocess", MyTransformer),
    ("model", model_XGB),
])

param_grid_xgb = {
    "model__regressor__n_estimators": [800, 1200],
    "model__regressor__learning_rate": [0.015, 0.025],
    "model__regressor__max_depth": [2, 3],
    "model__regressor__subsample": [0.75, 0.90],
    "model__regressor__colsample_bytree": [0.65, 0.80],
    "model__regressor__min_child_weight": [1, 3],
    "model__regressor__reg_lambda": [1.0],
    "model__regressor__reg_alpha": [0.0],
}

grid_xgb = GridSearchCV(
    estimator=Pipeline_XGB,
    param_grid=param_grid_xgb,
    scoring="neg_root_mean_squared_error",
    cv=kf,
    refit=True,
    n_jobs=-1,
    verbose=1,
)

grid_xgb.fit(X_train, y_train)

print("Best XGBoost parameters:")
print(grid_xgb.best_params_)
print("Best XGBoost CV RMSE:", -grid_xgb.best_score_)

best_xgb = grid_xgb.best_estimator_


## Comparing model validation scores

In this cell, we compare the cross-validated RMSE of Linear Regression, Random Forest, and XGBoost.

Lower RMSE is better. Since all models are evaluated with the same folds and the same preprocessing structure, this comparison is meaningful.


In [ ]:
model_names = ["Linear Regression", "Random Forest", "XGBoost"]
rmse_values = [
    -Hyperparameter_tuning_LR.best_score_,
    -Hyperparameter_tuning_RFR.best_score_,
    -grid_xgb.best_score_,
]

plt.figure(figsize=(8, 5))
plt.bar(model_names, rmse_values, edgecolor="black", color="tab:red")
plt.ylabel("Average CV RMSE")
plt.title("Model comparison: 5-fold CV RMSE")
plt.xticks(rotation=20)
plt.show()

comparison = pd.DataFrame({
    "Model": model_names,
    "CV_RMSE": rmse_values,
}).sort_values("CV_RMSE")

display(comparison)


## Out-of-fold predictions for the best XGBoost model

In this cell, we use `cross_val_predict` to generate out-of-fold predictions for the selected XGBoost pipeline.

Each training row is predicted by a model that did not train on that row. This makes the scatter plot more realistic than plotting fitted values from the final model.

The diagonal dashed line represents perfect predictions. Points close to the line indicate good predictions, while points far from the line indicate larger errors.


In [ ]:
y_pred_cv_xgb = cross_val_predict(
    best_xgb,
    X_train,
    y_train,
    cv=kf,
    n_jobs=-1,
)

rmse_oof_xgb = root_mean_squared_error(y_train, y_pred_cv_xgb)
print("XGBoost OOF RMSE:", rmse_oof_xgb)

plt.figure(figsize=(7, 7))
plt.scatter(y_train, y_pred_cv_xgb, alpha=0.5, color="green")

min_val = min(y_train.min(), y_pred_cv_xgb.min())
max_val = max(y_train.max(), y_pred_cv_xgb.max())

plt.plot([min_val, max_val], [min_val, max_val], linestyle="--", color="black")
plt.xlabel("Real SalePrice")
plt.ylabel("Predicted SalePrice")
plt.title("XGBoost cross-validated predictions vs real values")
plt.show()


## Optional ExtraTrees check

In this cell, we optionally test ExtraTrees as another tree-based model.

ExtraTrees is similar to Random Forest, but it uses more randomization when building tree splits. Sometimes this helps reduce overfitting. This cell is not the main final model, but it gives us another useful comparison.


In [ ]:
model_ETR = TransformedTargetRegressor(
    regressor=ExtraTreesRegressor(random_state=42, n_jobs=-1),
    func=np.log1p,
    inverse_func=np.expm1,
    check_inverse=False,
)

MyPipeline_ETR = Pipeline([
    ("preprocess", MyTransformer),
    ("model", model_ETR),
])

parameters_grid_ETR = {
    "model__regressor__n_estimators": [500, 900],
    "model__regressor__max_features": [0.5, 0.8],
    "model__regressor__min_samples_leaf": [1, 2],
    "model__regressor__min_samples_split": [2, 5],
}

Hyperparameter_tuning_ETR = GridSearchCV(
    estimator=MyPipeline_ETR,
    param_grid=parameters_grid_ETR,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    refit=True,
    n_jobs=-1,
    verbose=1,
)

Hyperparameter_tuning_ETR.fit(X_train, y_train)

print("Best ExtraTrees parameters:")
print(Hyperparameter_tuning_ETR.best_params_)
print("Best ExtraTrees CV RMSE:", -Hyperparameter_tuning_ETR.best_score_)

best_etr = Hyperparameter_tuning_ETR.best_estimator_


## Creating the final submission file

In this final cell, we use the best XGBoost pipeline to predict the sale prices for `X_test`.

Because `GridSearchCV` was created with `refit=True`, `best_xgb` has already been refitted on the full cleaned training set. Therefore, we can directly call `predict(X_test)`.

The predictions are already in the original SalePrice scale because `TransformedTargetRegressor` automatically applies the inverse transformation.

Finally, we create a CSV file with `Id` and `SalePrice`, which is the format needed for submission.


In [ ]:
test_predictions_xgb = best_xgb.predict(X_test)

submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": test_predictions_xgb,
})

submission.to_csv("francesco_maria_arezzo_ames_xgboost_predictions.csv", index=False)

display(submission.head())
print("Saved file: francesco_maria_arezzo_ames_xgboost_predictions.csv")
